In [3]:
import types
import torch
from augmult.augmentations import get_transforms_from_str
from augmult.data import dp_dataloader, get_dataset, init_mv_collate, non_dp_tokenize_dataloader
import torch.optim as optim
from opacus.utils.batch_memory_manager import BatchMemoryManager
from opacus import PrivacyEngine

from util.different_finetune_modes import get_param_setter_from_str, model_and_tokenizer
from util.logging_util import aug_name
from util.privacy_engine_util import add_noise, empty_batch_handling, prepare_gradsamplers
import os


In [4]:


MODEL_NAME = "bert-base-uncased"
EPOCHS = 2
BATCH_SIZE = 100
DATASETSIZE = None
LR = 1
TASK = "mednli"
GLUE = False
PRECOMP = True
NUM_LABELS = 3

os.environ["DATASET_DIR"] = "./datasets/processed"

In [5]:
trainable_param_setter = get_param_setter_from_str("classifier_and_pooler")
model, tokenizer = model_and_tokenizer(MODEL_NAME, NUM_LABELS)
trainable_param_setter(model)
optimizer = optim.SGD(model.parameters(), lr=LR)

# Dataloaders
dataset = get_dataset(TASK, glue = GLUE,precomputed_augs=PRECOMP)
mv_train_loader = dp_dataloader(dataset["train"],DATASETSIZE,tokenizer,[],BATCH_SIZE)
valid_loader = non_dp_tokenize_dataloader(dataset['validation'], tokenizer, 4096)

privacy_engine = PrivacyEngine()
dp_model, dp_optimizer, dp_train_loader = privacy_engine.make_private(
    module=model,
    optimizer=optimizer,
    data_loader=mv_train_loader,
    noise_multiplier=1.0,
    max_grad_norm=1.0
)
dp_optimizer.add_noise = types.MethodType(add_noise, dp_optimizer)
# Custom Grad Samplers
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dp_model.to(device=device)

pass

/home/spelz/miniconda3/envs/tan/lib/python3.9/site-packages/huggingface_hub-0.24.6-py3.8.egg/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/spelz/miniconda3/envs/tan/lib/python3.9/site-packages/opacus/privacy_engine.py:95: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


In [6]:
#transform_list = a.eda(5)
#transform_list = [a.unaugmented,a.context_insert]
#transform_list = ["unaug","zh","de","ru","ar"]
from augmult.augmentations import get_transforms_from_str

if not PRECOMP: transform_list = get_transforms_from_str(None)
else: transform_list = get_transforms_from_str("precomputed_12x1")

precomp= None in transform_list
K = len(transform_list)
mv_collate = init_mv_collate(tokenizer,transform_list,max_length=128,precomputed=precomp)
empty_batch_handling(mv_train_loader=mv_train_loader,dp_train_loader=dp_train_loader,mv_collate=mv_collate)


In [7]:
samples =[]
# BatchMemoryManager for handling large batches safely
with BatchMemoryManager(data_loader=dp_train_loader, max_physical_batch_size=512, optimizer=dp_optimizer) as memory_safe_data_loader: 
    for batch in memory_safe_data_loader:  # Iterating through augmented batches
        print(f"\n NEW BATCH"+"="*100+"\n")

        print(f"stacked views batch: {batch.keys()} | batchsize: {len(batch['input_ids'])}")

        for sample in batch['input_ids']:
            print(f"\n new sample: -------------------------------------\n")
            # Convert tokens back to text and print them
            for aug,augname in zip(sample,transform_list): #aug_name()
                example_to_string = tokenizer.decode(aug, skip_special_tokens=True,clean_up_tokenization_spaces=True)
                samples.append(example_to_string)
                print(f"{augname}:\n{example_to_string}\n")
        break

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



 NEW BATCH====================================================================================================

stacked views batch: dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels']) | batchsize: 91

 new sample: -------------------------------------

<function get_transforms_from_str.<locals>.<lambda> at 0x7c9db8d5e280>:
laboratories were remarkable for cr 1. 7 ( base 0. 5 per old record ) and lactate 2. 4. the patient has increased cr

None:
labs were notable for cr 1. 7 ( baseline 0. 5 per old records ) and lactate 2. 4. patient has elevated cr

None:
lab's famous for cr 1. 7 ( baseline 0. 5 per old record ) and the 2. 4. patients have increased cr

None:
labs was known for cr 1. 7 ( base 0. 5 per old record ) and lactate 2. 4. patience has cr.

None:
the laboratories were remarkable for cr 1. 7 ( base 0. 5 by old records ) and lactate 2. 4. the patient has a high rate of cr

None:
labs were notable for cr 1. 7 ( baseline 0. 5 per old records ) and lactate 2. 4

In [4]:
import nlpaug.augmenter.word as naw

#translate = Augmentations(emb=False,bert=False).back_translate
#t_cn = naw.BackTranslationAug(from_model_name='Helsinki-NLP/opus-mt-en-zh',to_model_name='Helsinki-NLP/opus-mt-zh-en',device="cuda",name="zh",).augment
#t_de = naw.BackTranslationAug(from_model_name='Helsinki-NLP/opus-mt-en-de',to_model_name='Helsinki-NLP/opus-mt-de-en',device="cuda",name="de",).augment
#t_ru = naw.BackTranslationAug(from_model_name='Helsinki-NLP/opus-mt-en-ru',to_model_name='Helsinki-NLP/opus-mt-ru-en',device="cuda",name="ru",).augment
#t_ar = naw.BackTranslationAug(from_model_name='Helsinki-NLP/opus-mt-en-ar',to_model_name='Helsinki-NLP/opus-mt-ar-en',device="cuda",name="ar",).augment
bt_aug_af = naw.BackTranslationAug(from_model_name='Helsinki-NLP/opus-mt-en-af',to_model_name='Helsinki-NLP/opus-mt-af-en',device="cuda",name="af",).augment
bt_aug_it = naw.BackTranslationAug(from_model_name='Helsinki-NLP/opus-mt-en-it',to_model_name='Helsinki-NLP/opus-mt-it-en',device="cuda",name="it",).augment
bt_aug_fr = naw.BackTranslationAug(from_model_name='Helsinki-NLP/opus-mt-en-fr',to_model_name='Helsinki-NLP/opus-mt-fr-en',device="cuda",name="fr",).augment
bt_aug_es = naw.BackTranslationAug(from_model_name='Helsinki-NLP/opus-mt-en-es',to_model_name='Helsinki-NLP/opus-mt-es-en',device="cuda",name="es",).augment
bt_aug_id = naw.BackTranslationAug(from_model_name='Helsinki-NLP/opus-mt-en-id',to_model_name='Helsinki-NLP/opus-mt-id-en',device="cuda",name="id",).augment
bt_aug_nl = naw.BackTranslationAug(from_model_name='Helsinki-NLP/opus-mt-en-nl',to_model_name='Helsinki-NLP/opus-mt-nl-en',device="cuda",name="nl",).augment
bt_aug_fi = naw.BackTranslationAug(from_model_name='Helsinki-NLP/opus-mt-en-fi',to_model_name='Helsinki-NLP/opus-mt-fi-en',device="cuda",name="fi",).augment


tl=[#t_cn,t_de,t_ru,t_ar,
bt_aug_af,
bt_aug_it,
bt_aug_fr,
bt_aug_es,
bt_aug_id,
bt_aug_nl,
bt_aug_fi,

]



In [ ]:
from augmult.augmentations import _load_emb
import nlpaug.augmenter.word as naw
import os
os.environ["MODEL_DIR"] = "/home/spelz/AugMult_DP_NLP/augmentation_models/"


word_params = {"aug_p":0.1,"aug_max":10,}
emb_param = {"top_k": 1,**word_params}

swap_word = naw.RandomWordAug(action="swap",**word_params,name="word_swap").augment
del_word = naw.RandomWordAug(**word_params,name="word_delete").augment
synonym_wn = naw.SynonymAug(aug_src='wordnet',name="wordnet_replace").augment
p = _load_emb()
glove_replace = naw.WordEmbsAug(**emb_param, model_path=p["glove"] ,model_type='glove',action="substitute",name="glove_replace").augment
glove_insert = naw.WordEmbsAug(**emb_param, model_path=p["glove"], model_type='glove',action="insert",name="glove_insert").augment
emb_replace = naw.WordEmbsAug(**emb_param, model_path=p["word2vec"] ,model_type='word2vec',action="substitute",name="w2v_replace").augment
emb_insert = naw.WordEmbsAug(**emb_param, model_path=p["word2vec"], model_type='word2vec',action="insert",name="w2v_insert").augment

tl = [swap_word,del_word,synonym_wn ,glove_replace,glove_insert,emb_replace,emb_insert]

In [11]:
#text = "When you've got snow, it's really hard to learn a snow sport so we looked at all the different ways I could mimic being on snow without actually being on snow."
#text = "i burst through a set of cabin doors, and fell to the ground - i burst through the doors and fell down."
#text = "Recurrent pleural effusion- unclear etiology, cytology negative in the past."
few_samples = samples[30:50]
os.environ["TOKENIZERS_PARALLELISM"] = "false"
for text in few_samples:
    print("\n---\n"+text)
    for t in tl:
        print(t.__self__.name,": ",t(text))


---
later in the day, the patient became progressively more agitated requiring increasing oxygen and sedative medications, and ultimately, she has to be reintubated. the patient is agitated
af :  ['Later in the day, the patient became more and more upset because he needed more and more oxygen and tranquilizers, and she eventually had to be reinvented.']
it :  ['later in the day, the patient became progressively more agitated requiring increase of oxygen and sedatives medications, and, finally, should be reintubated. the patient is agitated']
fr :  ['later in the day, the patient gradually became more agitated requiring an increase in oxygen and sedative medicines, and finally, she must be reincubated. the patient is agitated']
es :  ['Later in the day, the patient became progressively more agitated, requiring increased oxygen and sedative medications, and ultimately has to be reintubated.']
id :  ['Then later that day, the patient became increasingly restless, needing oxygen medicatio

In [17]:
len(transform_list)

12